In [ ]:
!apt-get install openjdk-17-jre -y
!pip3 install tweet-preprocessor
!pip3 install language-tool-python
!pip3 install contractions
!python3 -m spacy download en_core_web_sm
!pip3 install stanza requests

In [ ]:
import os
import re
import json
import time
import http.client
import sqlite3
import requests
import contractions
import spacy
import tweepy
import preprocessor as p
import language_tool_python

from typing import Callable, Optional, List, Tuple, Dict
from datetime import datetime, timedelta
from transformers import AutoTokenizer, AutoModelForSequenceClassification

reference_date = datetime(2025, 11, 4)

eng_tool = language_tool_python.LanguageTool('en-US')

In [ ]:
from google.colab import userdata

rapid_api = userdata.get('RapidAPI')
claimbuster_api = userdata.get('ClaimBusterAPI')

In [ ]:
# Module: claimbuster_post_scoring

def score_post_claimbuster(text: str, api_key: Optional[str] = None, timeout: int = 10):

    api_endpoint = "https://idir.uta.edu/claimbuster/api/v2/score/text/"
    request_headers = {"x-api-key": claimbuster_api}
    payload = {"input_text": text}
    api_response = requests.post(url=api_endpoint, json=payload, headers=request_headers, timeout=timeout)
    api_response.raise_for_status()
    return api_response.json()


In [ ]:
# Module: tweet_preprocessing

def preprocess_tweet(text: str) -> str:

    p.set_options(p.OPT.URL, p.OPT.RESERVED, p.OPT.EMOJI, p.OPT.SMILEY, p.OPT.HASHTAG)
    return p.clean(text)

def grammar_check(text: str) -> str:

    matches = eng_tool.check(text)
    corrected_text = language_tool_python.utils.correct(text, matches)
    return corrected_text


In [ ]:
#Temporal Normalization
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@([A-Za-z0-9_]{1,15})")


PHRASE_TO_DELTA = [
    (re.compile(r"\bday after tomorrow\b", flags=re.IGNORECASE), 2),
    (re.compile(r"\bday before yesterday\b", flags=re.IGNORECASE), -2),
    (re.compile(r"\byesterday\b", flags=re.IGNORECASE), -1),
    (re.compile(r"\blast night\b", flags=re.IGNORECASE), -1), 
    (re.compile(r"\btoday\b", flags=re.IGNORECASE), 0),
    (re.compile(r"\btomorrow\b", flags=re.IGNORECASE), 1),
]


EXTRA_PATTERNS = [
    (re.compile(r"\bthe day after tomorrow\b", flags=re.IGNORECASE), 2),
    (re.compile(r"\bthe day before yesterday\b", flags=re.IGNORECASE), -2),
]

ALL_PATTERNS = EXTRA_PATTERNS + PHRASE_TO_DELTA

def _mask_spans(text: str, spans: List[Tuple[int,int]]) -> str:
    if not spans:
        return text
    s = list(text)
    for a,b in spans:
        for i in range(a, b):
            s[i] = " "
    return "".join(s)

def simple_temporal_resolver(
    text: str,
    reference_date: Optional[datetime] = None,
    replace_in_text: bool = True
) -> Tuple[str, List[Tuple[str,str]]]:

    if reference_date is None:
        reference_date = datetime.utcnow()

    url_spans = [(m.start(), m.end()) for m in URL_RE.finditer(text)]
    masked = _mask_spans(text, url_spans)

    replacements: List[Tuple[str,str]] = []

    scheduled_replacements: List[Tuple[int,int,str,str]] = [] 

    for pattern, delta_days in ALL_PATTERNS:
        for m in pattern.finditer(masked):
            start, end = m.start(), m.end()
            overlap = False
            for (s,e,_,_) in scheduled_replacements:
                if not (end <= s or start >= e):
                    overlap = True
                    break
            if overlap:
                continue
            matched_text = text[start:end]  
            iso_date = (reference_date + timedelta(days=delta_days)).date().isoformat()
            scheduled_replacements.append((start, end, matched_text, iso_date))
            replacements.append((matched_text, iso_date))

    if not replace_in_text or not scheduled_replacements:
        return text, replacements

    scheduled_replacements.sort(key=lambda x: x[0])
    out_parts = []
    last = 0
    for start, end, orig, iso in scheduled_replacements:
        out_parts.append(text[last:start])
        out_parts.append(iso)
        last = end
    out_parts.append(text[last:])
    new_text = "".join(out_parts)

    return new_text, replacements

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Module: handle_cache_db

DB_SCHEMA = """
CREATE TABLE IF NOT EXISTS handle_cache (
    handle TEXT PRIMARY KEY,
    real_name TEXT,
    last_updated INTEGER
);
"""

def init_db(db_path: str = "handle_cache.db"):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.executescript(DB_SCHEMA)
    conn.commit()
    return conn

def get_cached_name(conn: sqlite3.Connection, handle: str) -> Optional[str]:
    cur = conn.cursor()
    cur.execute("SELECT real_name FROM handle_cache WHERE handle = ?", (handle.lower(),))
    row = cur.fetchone()
    return row[0] if row else None

def set_cached_name(conn: sqlite3.Connection, handle: str, real_name: str):
    cur = conn.cursor()
    cur.execute(
        "INSERT OR REPLACE INTO handle_cache(handle, real_name, last_updated) VALUES(?,?,strftime('%s','now'))",
        (handle.lower(), real_name)
    )
    conn.commit()


In [ ]:
# Module: rapidapi_lookup

def lookup_handle_via_rapidapi(
    handle: str,
    rapidapi_key: str,
    rapidapi_host: str = "twitter241.p.rapidapi.com",
    timeout: int = 10
) -> Tuple[bool, Optional[str], Optional[str]]:

    print(f"API CALL made for @{handle}")
    try:
        conn = http.client.HTTPSConnection(rapidapi_host, timeout=timeout)
        headers = {
            'x-rapidapi-key': rapidapi_key,
            'x-rapidapi-host': rapidapi_host
        }
        safe_handle = handle.lstrip("@")
        conn.request("GET", f"/user?username={safe_handle}", headers=headers)
        res = conn.getresponse()
        data = res.read()
        text = data.decode("utf-8")
        if res.status != 200:
            return False, None, f"HTTP {res.status}: {text}"
        payload = json.loads(text)
        user_info = payload.get("result", {}).get("data", {}).get("user", {}).get("result", {})
        if not user_info:
            return False, None, "unexpected payload structure"
        core = user_info.get("core", {})
        name = core.get("name")
        print("name given by rapid boi: "+name)
        if name:
            return True, name, None
        legacy_name = user_info.get("legacy", {}).get("name")
        if legacy_name:
            return True, legacy_name, None
        return False, None, "no name found in response"
    except Exception as e:
        return False, None, str(e)


In [ ]:
# This part of the code stores Handles/Names in Database to reduce api calls to replace mentions with actual names

def lookup_handle_via_rapidapi(
    handle: str,
    rapidapi_key: str,
    rapidapi_host: str = "twitter241.p.rapidapi.com",
    timeout: int = 10
) -> Tuple[bool, Optional[str], Optional[str]]:

    print(f"API CALL made for @{handle}")
    try:
        conn = http.client.HTTPSConnection(rapidapi_host, timeout=timeout)
        headers = {
            'x-rapidapi-key': rapidapi_key,
            'x-rapidapi-host': rapidapi_host
        }
        safe_handle = handle.lstrip("@")
        conn.request("GET", f"/user?username={safe_handle}", headers=headers)
        res = conn.getresponse()
        data = res.read()
        text = data.decode("utf-8")
        if res.status != 200:
            return False, None, f"HTTP {res.status}: {text}"
        payload = json.loads(text)
        user_info = payload.get("result", {}).get("data", {}).get("user", {}).get("result", {})
        if not user_info:
            return False, None, "unexpected payload structure"
        core = user_info.get("core", {})
        name = core.get("name")
        if name:
            return True, name, None
        legacy_name = user_info.get("legacy", {}).get("name")
        if legacy_name:
            return True, legacy_name, None
        return False, None, "no name found in response"
    except Exception as e:
        return False, None, str(e)

MENTION_RE = re.compile(r"@([A-Za-z0-9_]{1,15})")

def replace_mentions_with_names(
    text: str,
    rapidapi_key: str,
    db_path: str = "handle_cache.db",
    fallback_append: bool = False
) -> str:

    conn = init_db(db_path)
    mentions = MENTION_RE.findall(text)
    if not mentions:
        return text

    mapping = {}
    for h in set(mentions):
        cached = get_cached_name(conn, h)
        if cached:
            mapping[h] = cached
            continue
        ok, name, err = lookup_handle_via_rapidapi(h, rapidapi_key)
        if ok and name:
            set_cached_name(conn, h, name)
            mapping[h] = name
        else:
            mapping[h] = f"@{h} (name unavailable)" if fallback_append else f"@{h}"

    def _repl(match):
        handle = match.group(1)
        return mapping.get(handle, match.group(0))

    new_text = re.sub(MENTION_RE, _repl, text)
    conn.close()
    return new_text

In [ ]:
# Module: preprocessing_pipeline

def preprocessing(tweet_text: str, ref_date: datetime) -> str:
    # Claim Worthiness gate
    claim_worthiness = score_post_claimbuster(tweet_text)["results"][0]["score"]
    if claim_worthiness < 0.5:
        print("Post not worthy to be validated!")
        return tweet_text

    # 1) Remove hashtags/URLs/emojis/smileys
    text = preprocess_tweet(tweet_text)

    # 2) Grammar check
    text = grammar_check(text)

    # 3) Resolve @mentions to display names
    text = replace_mentions_with_names(text, rapid_api, fallback_append=True)

    # 4) Expand contractions
    text = contractions.fix(text)

    # 5) Temporal normalization (limited phrases)
    text, _ = simple_temporal_resolver(text, ref_date)
    return text



In [ ]:
# Module: related_news_fetch

API_BASE = "https://zliff-news-article-retrieval.hf.space"

def fetch_related_news(post_text: str, max_articles: int = 5) -> List[Dict[str, str]]:

    url = f"{API_BASE}/articles"
    payload = {"text": post_text, "with_content": True}
    try:
        resp = requests.post(url, json=payload, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        if not isinstance(data, list):
            raise ValueError("Unexpected response shape")

        out: List[Dict[str, str]] = []
        for i, item in enumerate(data[:max_articles]):
            title = (item or {}).get("title") or "Untitled"
            url = (item or {}).get("url") or ""
            content = (item or {}).get("content") or ""
            source = (item or {}).get("source") or ""
            published = (item or {}).get("publishedAt") or ""
            out.append({
                "id": f"news_{i}",
                "title": title,
                "url": url,
                "source": source,
                "publishedAt": published,
                "content": content
            })
        return out
    except Exception as e:
        print(f"[Error] Failed to fetch related news: {e}")
        return []


In [ ]:
# Module: sentence_splitting

def split_into_sentences(text: str) -> List[str]:

    if not text:
        return []
    text = text.strip().replace("\r\n", "\n").replace("\r", "\n")
    split_pattern = re.compile(r'(?<=[\.\?\!\;\:\u2014\-])\s+(?=[A-Z0-9"\'(\[])')
    raw_parts = split_pattern.split(text)

    parts: List[str] = []
    for part in raw_parts:
        subparts = re.split(r'\n{2,}', part)
        for sp in subparts:
            s = sp.strip()
            if s:
                s = re.sub(r'\s+', ' ', s)
                parts.append(s)

    cleaned = [p.strip(" \t\n") for p in parts if p.strip()]
    return cleaned


In [ ]:
# Module: claimbuster_sentence_scoring

def score_sents_claimbuster(text: str, api_key: Optional[str] = None):

    api_endpoint = "https://idir.uta.edu/claimbuster/api/v2/score/text/sentences/"
    request_headers = {"x-api-key": claimbuster_api}
    payload = {"input_text": text}
    api_response = requests.post(url=api_endpoint, json=payload, headers=request_headers)
    api_response.raise_for_status()
    return api_response.json()


In [ ]:
# Module: claim_filtering

def filter_claim_sentences(article_text: str, threshold: float = 0.5) -> Tuple[List[str], Dict[str, float]]:
    sents = split_into_sentences(article_text)
    sent_scores: List[float] = []
    for s in sents:
        res = score_sents_claimbuster(s)
        sent_scores.append(res["results"][0]["score"])
    sent_score_dict = {sents[i]: sent_scores[i] for i in range(len(sents)) if sent_scores[i] >= threshold}
    claims = list(sent_score_dict.keys())
    return claims, sent_score_dict

def merge_claims_text(claims: List[str]) -> str:
    return " ".join(claims).strip()


In [ ]:
# Module: lgbm_scoring

import numpy as np
from scipy.sparse import csr_matrix, hstack
import joblib
import lightgbm as lgb

loaded_model = joblib.load("/content/fake_news_lgbm.pkl")
tfv = joblib.load("/content/tfidf_vectorizer.pkl")

def score_post_with_article(article_title: str, article_text: str, post_title: str, post_body: str, model=loaded_model) -> float:

    atext = (str(article_title) + "\n" + str(article_text)).strip()
    ptext = (str(post_title) + "\n" + str(post_body)).strip()

    Xa = tfv.transform([atext])
    Xp = tfv.transform([ptext])

    dots = np.array((Xa.multiply(Xp)).sum(axis=1)).ravel()
    An = np.sqrt(np.array(Xa.power(2).sum(axis=1)).ravel()) + 1e-12
    Bn = np.sqrt(np.array(Xp.power(2).sum(axis=1)).ravel()) + 1e-12
    cos = (dots / (An * Bn)).astype(np.float32).reshape(-1, 1)

    Xf = hstack([Xp, csr_matrix(cos)], format="csr", dtype=np.float32)
    prob = float(model.predict_proba(Xf)[:, 1][0])
    return prob


In [ ]:
# Module: transformer_fact_checking

def transformer_fact_check_score(claim_text: str, evidence_text: str):
    max_len = alberta_tokenizer.model_max_length

    encoded_input = alberta_tokenizer.encode_plus(
        claim_text,
        evidence_text,
        return_tensors="pt",
        max_length=max_len,
        truncation=True
    )

    alberta_model.eval()
    with torch.no_grad():
        outputs = alberta_model(**encoded_input)

    logits = outputs.logits               
    probs = torch.softmax(logits, dim=1)  

    probs_np = probs.squeeze().cpu().numpy()  
    label = int(torch.argmax(probs, dim=1))    

    return label, probs_np

In [ ]:
# Module: cache_maintenance

def show_cache(db_path: str = "handle_cache.db"):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT handle, real_name, datetime(last_updated, 'unixepoch') FROM handle_cache")
    rows = cur.fetchall()
    conn.close()
    print("Cached handles:")
    for handle, real_name, updated in rows:
        print(f"@{handle} → {real_name} (cached at {updated})")


In [ ]:
# Module: colab_file_io

from google.colab import files

def export_cache_db(db_path: str = "handle_cache.db"):
    files.download(db_path)

def import_cache_db() -> None:
    uploaded = files.upload()


In [ ]:
# module: Routed Scoring

import numpy as np
from typing import Optional, Dict, Any, Tuple
from scipy.sparse import csr_matrix, hstack

# Your existing imports / assets
import joblib
import lightgbm as lgb


def _prep_text_pair(article_title: str, article_text: str,
                    post_title: str, post_body: str) -> Tuple[str, str, str, str, str, str, str]:
    a_title = str(article_title or "")
    a_body  = str(article_text or "")
    p_title = str(post_title or "")
    p_body  = str(post_body or "")

    atext = (a_title + "\n" + a_body).strip()
    ptext = (p_title + "\n" + p_body).strip()
    joint_for_transformer = f"[POST]\n{ptext}\n\n[ARTICLE]\n{atext}" 

    return a_title, a_body, p_title, p_body, atext, ptext, joint_for_transformer


def _tfidf_and_cosine(atext: str, ptext: str):
    Xa = tfv.transform([atext])
    Xp = tfv.transform([ptext])

    dots = np.array((Xa.multiply(Xp)).sum(axis=1)).ravel()
    An = np.sqrt(np.array(Xa.power(2).sum(axis=1)).ravel()) + 1e-12
    Bn = np.sqrt(np.array(Xp.power(2).sum(axis=1)).ravel()) + 1e-12
    cos = (dots / (An * Bn)).astype(np.float32).reshape(-1, 1)

    Xf = hstack([Xp, csr_matrix(cos)], format="csr", dtype=np.float32)
    return Xf, float(cos[0, 0])


def _length_signals(text: str) -> Dict[str, int]:
    words = text.split()
    chars = len(text)
    return {"n_words": len(words), "n_chars": chars}


def _lgbm_predict_proba(Xf, model: lgb.Booster) -> float:
    p = float(model.predict_proba(Xf)[0, 0])
    return p


def _transformer_score(post_text: str, article_text: str) -> Optional[float]:
    try:
        label, probs = transformer_fact_check_score(post_text, article_text)
    except Exception:
        return None
    if probs is None or len(probs) == 0:
        return None
    return float(probs[0])  



def _is_uncertain(p: float, band: float) -> bool:
    return (0.5 - band) < p < (0.5 + band)


def score_post_router(
    article_title: str,
    article_text: str,
    post_title: str,
    post_body: str,
    model: Any = loaded_model,
    router_cfg: Optional[Dict[str, Any]] = None
) -> Dict[str, Any]:

    if router_cfg is None:
        router_cfg = {}

    cfg = {
        "cosine_low_thresh": 0.10,    
        "cosine_high_conf": 0.35,       
        "len_min_words": 6,             
        "len_max_words": 250,           
        "uncertainty_band": 0.15,       
        "transformer_conf_margin": 0.10,
        "blend_weight_transformer": 0.65
    }
    cfg.update(router_cfg)

    # 1) Prep text
    a_title, a_body, p_title, p_body, atext, ptext, joint_text = _prep_text_pair(
        article_title, article_text, post_title, post_body
    )

    # 2) Sparse features + cosine
    Xf, cosine = _tfidf_and_cosine(atext, ptext)

    # 3) LGBM probability (REAL)
    p_lgbm = _lgbm_predict_proba(Xf, model)

    # 4) Routing signals
    len_sig = _length_signals(ptext)
    n_words = len_sig["n_words"]

    route_reasons = []
    if cosine < cfg["cosine_low_thresh"]:
        route_reasons.append(f"low_cosine<{cfg['cosine_low_thresh']:.2f}")
    if n_words < cfg["len_min_words"] or n_words > cfg["len_max_words"]:
        route_reasons.append("length_out_of_range")
    if _is_uncertain(p_lgbm, cfg["uncertainty_band"]):
        route_reasons.append("lgbm_uncertain")

    use_transformer = len(route_reasons) > 0
    p_trf = None

    # 5) If routed, get transformer semantic probability
    if use_transformer:
        p_trf = _transformer_score(post_body, article_text)

    # 6) Decision policy
    if not use_transformer or p_trf is None:
        p_final = p_lgbm
        policy = "lgbm_only"
    else:
        trf_margin = abs(p_trf - 0.5)
        lgbm_margin = abs(p_lgbm - 0.5)

        if trf_margin >= cfg["transformer_conf_margin"] and (p_trf > 0.5) != (p_lgbm > 0.5):
            p_final = p_trf
            policy = "transformer_override"
        else:
            w = cfg["blend_weight_transformer"]
            p_final = float(w * p_trf + (1.0 - w) * p_lgbm)
            policy = "blended"

    label_final = "REAL" if p_final >= 0.5 else "FAKE"

    return {
        "p_final": p_final,
        "label_final": label_final,
        "p_lgbm": p_lgbm,
        "p_transformer": p_trf,
        "used_transformer": use_transformer,
        "signals": {
            "cosine": cosine,
            "n_words_post": n_words,
            "uncertain_lgbm": _is_uncertain(p_lgbm, cfg["uncertainty_band"]),
            "route_reasons": route_reasons
        },
        "policy": policy
    }


def score_post_with_router(
    article_title: str,
    article_text: str,
    post_title: str,
    post_body: str,
    model: Any = loaded_model,
    router_cfg: Optional[Dict[str, Any]] = None
) -> float:
    out = score_post_router(
        article_title=article_title,
        article_text=article_text,
        post_title=post_title,
        post_body=post_body,
        model=model,
        router_cfg=router_cfg
    )
    return float(out["p_final"])


In [ ]:
# Validate Tweet Here
tweet = ("""<Insert Tweet Here""")

if __name__ == "__main__":
    # 1) Preprocess post
    pre_processed_tweet = preprocessing(tweet, reference_date)
    print("Input to the Model:\nPreprocessed tweet:\n", pre_processed_tweet)

    # 2) Fetch related news
    articles = fetch_related_news(pre_processed_tweet, max_articles=3) or []
    for idx, article in enumerate(articles):
        print(f"[Article {idx+1}] URL:", article.get("url", ""))
        body = article.get("content", "") or ""

    # 3) Build claim text from best article (if any)
    comb_text = ""
    if articles:
        best_body = articles[0].get("content", "") or ""
        claims, sent_scores = filter_claim_sentences(best_body)
        comb_text = merge_claims_text(claims)
    print("Article fetched:\n" + (comb_text if comb_text else "(none)"))

    # 4) Score with router (LGBM + conditional transformer)
    if comb_text.strip():
        router_cfg = {
            "cosine_low_thresh": 0.30,
            "uncertainty_band": 0.25,
            "blend_weight_transformer": 0.85,
            "transformer_conf_margin": 0.10
        }

        out = score_post_router(
            article_title="",
            article_text=comb_text,
            post_title="",
            post_body=pre_processed_tweet,
            router_cfg=router_cfg
        )

        print("\n=== Router Decision ===")
        print(f"Final probability (REAL): {out['p_final']:.4f}")
        print(f"Final label: {out['label_final']}")
        print(f"LGBM prob (REAL): {out['p_lgbm']:.4f}")
        if out['p_transformer'] is not None:
            print(f"Transformer prob (REAL): {out['p_transformer']:.4f}")
        print(f"Used transformer: {out['used_transformer']}")
        print(f"Policy: {out['policy']}")
        print(f"Signals: cosine={out['signals']['cosine']:.4f}, "
              f"n_words_post={out['signals']['n_words_post']}, "
              f"uncertain_lgbm={out['signals']['uncertain_lgbm']}, "
              f"route_reasons={out['signals']['route_reasons']}")

    else:

        print("\nNo article content available; routing skipped.")